# <center/> Photometric Classifcation of Brown Drawfs </center>
#### <center/> ASTR 154 Final Project </center>
<center/> Marylin Loritsch and Lauren Agarwala </center>


In [5]:
# import statements
from astroquery.vizier import Vizier  # import Vizier to query AllWISE catalog
from astropy.coordinates import SkyCoord  # import SkyCoord to query tables by RA & DEC
import astropy.units as u  # for specifying units during coordinate query 
from astropy.table import Table, vstack  # vstack for combining l & t dwarf tables
import numpy as np
from astropy.modeling.models import BlackBody  # blackbody function for mcmc
import emcee  # import mcmc
import corner  # import for creating mcmc corner plot

General Description/Overview

# <center/> Data Sets </center>

Gen description


In [6]:
# The first step of our project is to use astroquery to query the Vizier data release AllWISE Source Catalog for our sources.
# We will start by obtaining the data for our sample of brown dwarfs, given in N. Skrzypek et al. (2016), 
# see: https://doi.org/10.1051/0004-6361/201527359 for the original article.
# Unfortunately this article only reports some of the WISE photometry, so we will query the AllWISE Source Catalog by coordinate to collect 
# information for these sources. Hence, we must collect the brown dwarf positions (RA & DEC) first from N. Skrzypek et al. (2016):

# Start by defining Vizier object:
v1 = Vizier(columns = ['**'])  # the '**' tells Vizier to list all columns in the catalog UPDATE PROB DONT NEED THIS
v1.ROW_LIMIT=-1  # this tells not to limit the rows returned 

# Next use Vizier.get_catalogs to query Vizier for the catlog we're interested in
bd_catalog = v1.get_catalogs('J/A+A/589/A49')  # J/A+A/589/A49 is the name of the catalog given by N. Skrzypek et al. (2016)

# This catalog contains 3 tables, one with information for the L-dwarfs identitfied in WISE, one for the T-dwarfs, and one for references
# We are interested in the tables containing information for the L & T dwarfs:
ldwarf_table = bd_catalog[0]  # L-dwarf table is 0th table in catalog
tdwarf_table = bd_catalog[1]  # T-dwarf table is 1st table in catalog

filters = ['Jmag', 'e_Jmag', 'Hmag', 'e_Hmag', 'Kmag', 'e_Kmag','W1mag', 'e_W1mag', 'W2mag', 'e_W2mag']  # list to specify columns we want

ldwarf_pos_table = ldwarf_table[filters]  # only grab filter columns from l dwarf table
tdwarf_pos_table = tdwarf_table[filters]  # only grab filter columns from t dwarf table



bd_phot_table = vstack([ldwarf_pos_table, tdwarf_pos_table], metadata_conflicts = 'silent')  # stack these tables together for one table!
bd_phot_table['Label'] = 'BD'  # add column labeling all of these sources as brown dwarf (BD)

# we only want sources with data in all filters! mask out those missing data: 
bd_phot_table = bd_phot_table.to_pandas()  # convert to pandas data frame, easier to work with for this
mask = bd_phot_table[filters].notna().all(axis = 1)  # .notna() grabs only not missing values. .all(axis = 1) checks every column in a row
bd_phot_table = bd_phot_table[mask]  # apply mask
bd_phot_table = Table.from_pandas(bd_phot_table)  # convert back to astropy table for stacking with NBD table later

bd_phot_table  # to check

Jmag,e_Jmag,Hmag,e_Hmag,Kmag,e_Kmag,W1mag,e_W1mag,W2mag,e_W2mag,Label
float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,str2
14.76,0.01,14.08,0.01,13.54,0.01,13.33,0.02,13.02,0.03,BD
15.46,0.01,14.48,0.01,13.62,0.01,12.97,0.02,12.54,0.02,BD
17.18,0.02,16.43,0.03,15.9,0.03,15.65,0.05,15.38,0.1,BD
16.59,0.01,15.94,0.01,15.36,0.01,15.06,0.04,14.8,0.07,BD
16.6,0.01,15.91,0.02,15.35,0.02,15.08,0.04,14.84,0.07,BD
16.95,0.02,16.27,0.02,15.75,0.02,15.4,0.04,15.2,0.09,BD
16.69,0.02,15.88,0.02,15.21,0.02,14.86,0.03,14.68,0.06,BD
16.93,0.02,16.26,0.03,15.7,0.02,15.34,0.04,15.07,0.08,BD
17.47,0.03,16.64,0.03,15.92,0.02,15.28,0.04,15.14,0.1,BD


In [8]:
v2 = Vizier(columns = filters)
v2.ROW_LIMIT= 10000
result = v2.query_constraints(catalog = 'II/328/allwise', 
                              ex = '0',  # indicates a point source
                              ccf = '0000',  # avoids contamination by various image artifacts                   
                              Jmag = '!null',  # only grabs data with Jmags
                              e_Jmag = '!null',  # & corresponding error
                              Hmag = '!null',  # only grabs data with Hmags
                              e_Hmag = '!null',  # & corresponding error
                              Kmag = '!null',  # only grabs data with Kmags
                              e_Kmag = '!null',  # & corresponding error
                              W1mag = '!null',  # only grabs data with W1mags
                              e_W1mag = '!null',  # & corresponding error
                              W2mag = '!null',  # only grabs data with W2mags
                              e_W2mag = '!null',  # & corresponding error
                             )  
nonbd_phot_table = result[0]

length = len(bd_phot_table)
random_i = np.random.choice(length, size = length, replace = False)
nonbd_phot_table = nonbd_phot_table[random_i]
nonbd_phot_table['Label'] = 'NBD'  # label for not brown dwarf
nonbd_phot_table  # to check

# NEED TO FINISH COMMENTING

Jmag,e_Jmag,Hmag,e_Hmag,Kmag,e_Kmag,W1mag,e_W1mag,W2mag,e_W2mag,Label
mag,mag,mag,mag,mag,mag,mag,mag,mag,mag,
float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,str3
4.609,0.348,3.774,0.256,3.619,0.350,3.447,0.404,3.166,0.335,NBD
4.680,0.212,3.901,0.234,3.779,0.290,3.705,0.390,3.555,0.252,NBD
4.843,0.198,3.751,0.212,3.502,0.260,3.442,0.388,2.947,0.374,NBD
4.968,0.037,3.671,0.212,3.228,0.258,3.691,0.408,3.540,0.282,NBD
4.801,0.296,4.042,0.226,4.027,0.260,3.848,0.363,3.584,0.197,NBD
4.468,0.202,3.893,0.228,3.855,0.276,3.737,0.370,3.468,0.241,NBD
4.661,0.260,3.619,0.230,3.285,0.312,3.202,0.491,2.870,0.326,NBD
4.593,0.206,3.715,0.240,3.626,0.230,3.625,0.415,3.426,0.285,NBD


In [9]:
# NEED TO COMMENT add tables & mix up the rows

table = vstack([bd_phot_table, nonbd_phot_table], metadata_conflicts = 'silent')

table_length = len(table)
table_random_i = np.random.choice(table_length, size = table_length, replace = False)
table = table[table_random_i]

table

Jmag,e_Jmag,Hmag,e_Hmag,Kmag,e_Kmag,W1mag,e_W1mag,W2mag,e_W2mag,Label
mag,mag,mag,mag,mag,mag,mag,mag,mag,mag,
float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,str3
4.558,0.198,3.729,0.162,3.520,0.210,3.495,0.447,3.333,0.257,NBD
17.140,0.020,16.360,0.030,15.770,0.020,14.980,0.040,14.520,0.060,BD
4.963,0.236,3.929,0.214,3.520,0.286,3.501,0.491,3.255,0.231,NBD
4.880,0.037,3.914,0.260,3.722,0.290,3.646,0.416,3.487,0.183,NBD
4.471,0.248,3.496,0.224,3.327,0.222,3.347,0.542,3.143,0.469,NBD
16.260,0.010,15.620,0.010,14.980,0.010,14.660,0.030,14.310,0.060,BD
17.340,0.030,16.740,0.030,16.150,0.030,15.840,0.050,15.700,0.150,BD
5.439,0.018,3.973,0.186,3.675,0.248,3.872,0.386,3.633,0.262,NBD


notes about the sets

 # <center/> Classification </center>

 general description on what it is, what it does, plan

In [15]:
# iris.target
# random forest; https://www.geeksforgeeks.org/machine-learning/random-forest-regression-in-python/; 
# https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html
# https://astroquery.readthedocs.io/en/latest/
from sklearn.ensemble import RandomForestRegressor
from astroquery.simbad import Simbad

label = table['Label'] # y vals

#Look into what iris returns for intepretability
print(label)
#classifier = RandomForestRegressor( 100, criterion = 'squared_error', oob_score = True)
#classifier.fit(table, 

Label
-----
  NBD
   BD
  NBD
  NBD
  NBD
   BD
   BD
  NBD
  NBD
   BD
   BD
   BD
  NBD
  ...
  NBD
  NBD
  NBD
  NBD
   BD
  NBD
   BD
  NBD
   BD
  NBD
   BD
  NBD
   BD
Length = 2540 rows


Notes about it

# <center/> MCMC </center>

general description and vibes; how it connects to classification


In [3]:
# these are for testing for now!
app_mag_vals = np.array([14.130, 13.420, 12.810, 12.580, 12.340]) # complete
e_app_mag_vals = np.array([0.010, 0.010, 0.010, 0.020, 0.020]) # complete
wavelength_vals = np.array([1.235, 1.662, 2.159, 3.4, 4.6])  # central wavelengths for J, H, K, W1, W2 filters in microns
F0_vals = np.array([1594, 1024, 666.7, 306.68, 170.66]) # zero point magnitude for each filter in microJ (this is for app mag --> flux conversion)

In [ ]:
def mag_to_flux(m, e_m, F0):
    F = F0 * 10**(-0.4*m)
    dfdm = -0.4 * np.log(10) * F
    variance_F = (dfdm**2)*(e_m**2)
    e_F = variance_F**(1/2)

    return F, e_F


# note that F_nu = alpha * B_nu where alpha = pi * (R/D)^2. So modeling flux with parameters alpha & temperature
def model_flux(theta, wave):
    T, alpha = theta
    wavelengths = wave * u.micron
    
    bb = BlackBody(temperature = T * u.K)
    B_nu = bb(wavelengths).to(u.Jy / u.sr)
    flux_model = alpha * B_nu.value

    return flux_model
    

# write functions
# F_nu = alpha * B_nu where alpha = pi * (R/D)^2
# need to figure out mags to F_nu

In [ ]:
# define max & min values for T & alpha
T_min = 500 # complete
T_max = 3000 # complete

# ASK ABOUT ALPHA --> you don't know nothing! use approx radius of these sources & mw distances
R_BD = 0.8 * u.R_jup
dist_min = 1 * u.pc
dist_max = 100000 * u.pc

alpha_min = ((R_BD / dist_max)**2).value
alpha_max = ((R_BD / dist_min)**2).value

def log_prior(theta):
    T_arr, alpha_arr = theta

    shape = np.shape(T_arr)
    T_fin_arr = np.full(shape, 0)
    alpha_fin_arr = np.full(shape, 0)

    T_match = (T_min <= T_arr) & (T_arr <= T_max)
    alpha_match = (alpha_min <= alpha_arr) & (alpha_arr <= alpha_max)

    T_fin_arr[T_match] = 1
    alpha_fin_arr[alpha_match] = 1

    prior = T_fin_arr * alpha_fin_arr
    ln_prior = np.log(prior)

    return ln_prior
    

def log_likelihood(theta, app_mag, e_app_mag, F0, wave):  # NEED TO CHECK AXIS STUFF
    temp, a = theta
    T = temp[...,np.newaxis]
    alpha = a[..., np.newaxis]

    flux, e_flux = mag_to_flux(app_mag, e_app_mag, F0)
    mu = model_flux((T, alpha), wave)

    ln_quant = np.log(2 * np.pi * e_flux**2)
    sq_quant = ((flux - mu) / e_flux)**2
    lnL_arr = -0.5 * (ln_quant + sq_quant)
    lnL = np.sum(lnL_arr, axis = -1)
    return lnL


def log_posterior(theta, app_mag, e_app_mag, F0, wave):
    ln_posterior = log_prior(theta) + log_likelihood(theta, app_mag, e_app_mag, F0, wave)

    return ln_posterior

In [ ]:
# MCMC

# initialize params
n_params = 2  # number of parameters
n_walkers = 100  # number of walkers
n_steps = 10000  # number of steps

# pick starting positions
T_random = np.random.uniform(low = T_min, high = T_max, size = n_walkers)  # for each walker draw numb between T min & max
alpha_random = np.random.uniform(low = alpha_min, high = alpha_max, size = n_walkers)  # do the same for alpha
initial_guesses = np.array([T_random, alpha_random]).T  # array combining initial guesses for both parameters 
                                                        # note must transpose from (ndim, nwalkers) --> (nwalkers, ndim)
                                                        # so each row is a singler walker's T & alpha values

In [ ]:
# initialize & run

# initialize sampler
n_dim = np.shape(initial_guesses)[1]  # recall initial_guesses has shape (nwalkers, ndim)

# initialize EnsembleSampler with number of walkers, number of dimensions, and the log_posterior function
sampler = emcee.EnsembleSampler(
                    nwalkers = n_walkers,
                    ndim = n_dim,
                    log_prob_fn = log_posterior,
                    args = [app_mag_vals, e_app_mag_vals, F0_vals, wavelength_vals] 
                    )

# run 
sampler.run_mcmc(initial_guesses, n_steps)

notes


Ending thoughts